# Module 2 — Using Claude Code for String + File Processing

**SBML Lab Intern Training**

---

This module shows how Claude Code can generate file-processing and data-manipulation scripts from natural language descriptions.

The core skill is simple:

> **Describe what you want in plain English → get a script → refine it with follow-ups.**

By the end of this module you will have practiced this loop on GFF file parsing and pandas workflows — two tasks you will encounter constantly in the lab.


## Background: Genome Annotations and the GFF Format

### What is a genome annotation?

A genome is just a long string of DNA — billions of base pairs with no obvious structure by itself. A **genome annotation** is a map that labels where things are: this region is a gene, this region encodes a protein. Without an annotation, you have sequence but no biology.

The lab's annotation for *E. coli* K-12 MG1655 marks every gene in the genome — about 4,600 features total (all rows are `gene` features; this file has no separate `CDS` rows). This file is the reference you'll work with throughout the training.

### The GFF format

GFF (General Feature Format) is the standard file format for genome annotations. Every row describes one genomic feature (a gene, CDS, etc.) using 9 tab-separated columns:

The columns below describe the GFF format in general; the **Example** column shows the actual values found in this lab annotation file:

| Column | Name | Example (this file) |
|--------|------|---------|
| 1 | Chromosome / sequence name | `NC_000913` |
| 2 | Source | `Ecocyc_14.1` |
| 3 | Feature type | `gene` |
| 4 | Start position | `337` |
| 5 | End position | `2799` |
| 6 | Score | `.` (unused here) |
| 7 | Strand | `+` or `-` |
| 8 | Phase | `.` |
| 9 | Attributes | `locus_tag=b0002;gene=thrA;product=...;color=00FF00` |

> **Note the attribute keys:** this file uses `locus_tag=` and `gene=` (not `ID=`/`Name=`). Parse for those keys when extracting gene names.
>
> **Sequence-name convention:** this annotation uses `NC_000913` (no version suffix). The reference genome you download in Module 3 is `NC_000913.3`. The suffix differs — this matters when you join files by chromosome (Module 4), where `NC_000913` and `NC_000913.3` will not match unless you reconcile them.

**Coordinates are 1-based** — position 1 is the first nucleotide of the chromosome. This matters when writing Python code, since Python string indexing is 0-based. An off-by-one error here silently shifts every position by one.

### Why does this lab use GFF?

GFF is the input format for **MetaScope**, the lab's genome browser for visualizing ChIP-exo data. The alignment pipeline (Module 3) produces a GFF file as its final output so reads can be loaded into MetaScope alongside the gene annotation.

---

## GFF Format Recap

GFF is the standard annotation format used in this lab — tab-delimited, 9 columns, no header row, 1-based coordinates. Use `lab-context.md` as a reference if needed.

**Data for this module:** the lab *E. coli* K-12 MG1655 annotation GFF is pre-provided at:
```
data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff
```
This is the same file used in Modules 4 and 5 (the mini-project).


## Natural Language → Script

The pattern you will use throughout this module:

1. **Write a plain-English description** of what you want the script to do. Be specific: mention the file format, the columns involved, and the expected output.
2. **Claude Code generates a first draft.** Paste it into a code cell and run it.
3. **Follow up with refinements.** Fix edge cases, add output formatting, handle errors, change the logic.

> **Important:** your first prompt is a draft, not a final answer. Follow-up prompts are where the real work happens. Most production scripts in the lab went through 3–10 iterations before they were committed.

### Tips for better first prompts

- Name the file format explicitly: *"a GFF file"*, not *"a text file"*.
- State the output: *"print the result to stdout"* or *"save to a new TSV file"*.
- Mention what to do with edge cases: *"skip comment lines"*, *"ignore rows where score is `.`"*.

---

### Why would you filter a genome annotation?

Real analyses rarely use the entire annotation. Common filtering scenarios in this lab:

- **By strand:** ChIP-exo reads are strand-specific — a binding site on the `+` strand regulates genes on the `+` strand. You often want to look at only one strand at a time.
- **By feature type:** You might want only `gene` entries, not `CDS`, depending on the question.
- **By coordinate range:** If you're zooming in on a genomic region (say, the first 500 kb), you only want features within those coordinates.

### What does "strand" mean?

The *E. coli* chromosome is double-stranded DNA. Genes can be encoded on either strand:

- **`+` strand (forward):** The gene is read left-to-right as written in the reference FASTA. Start < End.
- **`-` strand (reverse):** The gene is read right-to-left. In GFF, start is still less than end (by convention), but the strand column is `-`.

This matters biologically because a TF binding site upstream of a `+` strand gene is in a different genomic direction than one upstream of a `-` strand gene.

---

### Exercise 1 — Generate and refine a GFF filtering script

1. Describe a GFF filtering task in plain English to Claude Code. Use the lab annotation GFF at `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff`. For example: filter by feature type, by strand, or by coordinate range — your choice.
2. Generate the script.
3. Improve it with **at least two follow-up prompts**.

Write the **final version** of your script in the code cell below. Add a comment at the top of the script that briefly describes what each follow-up prompt changed.

Example comment format:
```python
# Follow-up 1: added argparse so the input path is a CLI argument
# Follow-up 2: changed output to TSV instead of printing to stdout
```

> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [ ]:
#!/usr/bin/env python3
# initial prompt used: make a GFF parser&filter python script. When it is runned, ask the user to enter the filepath of the .gff file. read the file, it should be consisted of 9 tab-delimited columns. to execute this script, for example, user should enter this line in CLI. python gff_filter.py /workspaces/intern-claude-training/data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff -featuretype "gene" -positionrange 500000-10000000 -strand "+" script should filter the lines and create a new GFF file in the same directory as the original GFF file, and the new file's name includes the filter options, for example, "ec_annotation_20100903_DHK_cSRNA_with_ortho.gene_500000-1000000_+.gff" . the new file should only include, for example, lines of which its feature type (column[2]) is gene, start position (column[3]) is more than 500000, end position (column[4]) is less than 1000000, strand(column[5]) is +. all the filter options are optional, but at least 1 filter should be implemented. if the input is for example "python gff_filter.py /workspaces/intern-claude-training/data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff", just show message that no filter is added. I didn't include the filter options for the attributes column([8]), suggest me how to make filter for attributes before you make script. If there are any other ideas of how to improve this script, suggest.
# 위의 Follow-up 예시를 보고 CLI에서 실행하는 GFF 필터링 스크립트를 작성했습니다.
# 실행예: python gff_filter.py -featuretype "gene" -positionrange 500000-10000000 -strand "+" -attribute "gene=thrA" -attribute "product~kinase" -attribute "KpOrtho"
# Follow-up 1: added NOT logic for each filter type, so you can exclude features by type, position range, strand, and/or attributes.
# Follow-up 2: added a check for non-numeric start/end coordinates when using position filters, and a warning if they are present.

"""Filter a 9-column GFF file by feature type, position range, strand, and/or attributes."""

import argparse
import os
import re
import sys

NUM_COLUMNS = 9
SEQID, SOURCE, TYPE, START, END, SCORE, STRAND, PHASE, ATTRIBUTES = range(NUM_COLUMNS)


def parse_args():   # argparse로 CLI인자를 정의하고 파싱한다. gff_filepath, featuretype, positionrange, strand, outputdir, attribute, exclude_featuretype, exclude_positionrange, exclude_strand, exclude_attribute를 인자로 받는다. outputdir는 미지정시 gff_filepath와 동일한 디렉토리로 설정된다. attribute와 exclude_attribute는 반복 가능하며, KEY=VALUE, KEY~SUBSTRING, KEY 형식으로 지정할 수 있다.
    parser = argparse.ArgumentParser(
        description="Filter a GFF file by feature type, position range, strand, and/or attributes."
    )
    parser.add_argument("gff_filepath", help="path to the input .gff file")
    parser.add_argument("-featuretype", help='feature type to keep, e.g. "gene" (column 3)')
    parser.add_argument(
        "-positionrange",
        help='keep features with start > MIN and end < MAX, e.g. "500000-1000000" (columns 4/5)',
    )
    parser.add_argument("-strand", choices=["+", "-"], help="strand to keep (column 7)")
    parser.add_argument(
        "-outputdir",
        help="directory to write the filtered GFF to (default: same directory as the input file)",
    )
    parser.add_argument(
        "-attribute",
        action="append",
        dest="attributes",
        metavar="KEY=VALUE|KEY~SUBSTRING|KEY",
        help=(
            "attribute filter on column 9, repeatable (AND-combined). "
            'KEY=VALUE for exact match, KEY~SUBSTRING for substring match, '
            "KEY alone to require the key to be present. "
            'Examples: -attribute "gene=thrA" -attribute "product~kinase" -attribute "KpOrtho"'
        ),
    )
    parser.add_argument("-exclude-featuretype", help="feature type to drop, e.g. \"tRNA\" (column 3)")
    parser.add_argument(
        "-exclude-positionrange",
        help='drop features with start > MIN and end < MAX, e.g. "500000-1000000" (columns 4/5)',
    )
    parser.add_argument("-exclude-strand", choices=["+", "-"], help="strand to drop (column 7)")
    parser.add_argument(
        "-exclude-attribute",
        action="append",
        dest="exclude_attributes",
        metavar="KEY=VALUE|KEY~SUBSTRING|KEY",
        help="attribute filter on column 9 to drop, repeatable (AND-combined). Same syntax as -attribute.",
    )
    return parser.parse_args()


def parse_positionrange(value): # "5000-10000" 형식의 문자열을 받아서 '-'로 split한다. 형식에 오류가 있으면 sys.exit()로 종료한다. split한 두 부분을 int로 변환하여 튜플로 반환한다.
    parts = value.split("-")
    if len(parts) != 2:
        sys.exit(f"Error: -positionrange must look like 'MIN-MAX', got '{value}'")
    try:
        return int(parts[0]), int(parts[1])
    except ValueError:
        sys.exit(f"Error: -positionrange must contain integers, got '{value}'")


def parse_attribute_filter(raw):    # -attribute로 입력받은 원본 문자열을 (key, 연산자, value) 튜플로 변환한다. KEY=VALUE, KEY~SUBSTRING, KEY 형식으로 지정할 수 있다. KEY=VALUE는 정확히 일치하는 경우, KEY~SUBSTRING은 부분 문자열이 포함되는 경우, KEY만 있는 경우는 해당 키가 존재하는 경우를 의미한다.
    if "~" in raw:
        key, _, value = raw.partition("~")
        return key.strip(), "~", value.strip()
    if "=" in raw:
        key, _, value = raw.partition("=")
        return key.strip(), "=", value.strip()
    return raw.strip(), "exists", None


def parse_attribute_column(text):   # GFF 9번째 column을 파싱해서 딕셔너리로 변환한다.
    attrs = {}
    for entry in text.split(";"):
        entry = entry.strip()
        if not entry:
            continue
        key, sep, value = entry.partition("=")
        attrs[key.strip().lower()] = value.strip() if sep else ""
    return attrs


def attribute_matches(attrs_lower, key, op, value): # 파싱된 attribute 딕셔너리에 대해 주어진 key, 연산자, value가 일치하는지 확인한다. key는 대소문자를 구분하지 않고 비교한다. op는 "exists", "=", "~" 중 하나이다. "exists"는 해당 key가 존재하는 경우를 의미하고, "="는 정확히 일치하는 경우를 의미하며, "~"는 부분 문자열이 포함되는 경우를 의미한다.
    key_lower = key.lower()
    if key_lower not in attrs_lower:
        return False
    if op == "exists":
        return True
    actual = attrs_lower[key_lower]
    if op == "=":
        return actual.lower() == value.lower()
    if op == "~":
        return value.lower() in actual.lower()
    return False


def sanitize(text): # 출력될 파일명에 쓸 수 없는 문자를 '_'로 바꾼다.
    return re.sub(r"[^A-Za-z0-9+\-]", "_", text)


def attribute_token(key, op, value):    # attribute 필터를 파일명 토큰으로 변환한다.
    return f"attr-{key}{op if op != 'exists' else ''}{value or ''}"


def build_output_path(  # 입력 시 gff_filepath와 필터옵션들을 받아서, 출력 파일의 경로를 조립한다.
    input_path,
    featuretype,
    positionrange,
    strand,
    attribute_filters,
    exclude_featuretype,
    exclude_positionrange,
    exclude_strand,
    exclude_attribute_filters,
    outputdir=None,
):
    directory, filename = os.path.split(input_path)
    if outputdir:
        directory = outputdir
    stem, ext = os.path.splitext(filename)

    parts = []
    if featuretype:
        parts.append(sanitize(featuretype))
    if positionrange:
        start, end = positionrange
        parts.append(f"{start}-{end}")
    if strand:
        parts.append(strand)
    for key, op, value in attribute_filters:
        parts.append(sanitize(attribute_token(key, op, value)))
    if exclude_featuretype:
        parts.append(sanitize(f"not-{exclude_featuretype}"))
    if exclude_positionrange:
        start, end = exclude_positionrange
        parts.append(f"not-{start}-{end}")
    if exclude_strand:
        parts.append(f"not-{exclude_strand}")
    for key, op, value in exclude_attribute_filters:
        parts.append(sanitize(f"not-{attribute_token(key, op, value)}"))

    suffix = "_".join(parts)
    return os.path.join(directory, f"{stem}.{suffix}{ext}")


def parse_coords(fields):   # GFF의 [4], [5] column을 int로 변환하여 튜플로 반환한다. 변환에 실패하면 None을 반환한다. (follow-up 2)
    try:
        return int(fields[START]), int(fields[END])
    except ValueError:
        return None


def line_matches(   # 핵심 필터링 로직으로, 한 줄이 모든 조건을 만족하는지 확인한다. 하나라도 조건을 만족하지 않으면 False를 반환한다. exclude 옵션이 있는 경우, 해당 조건을 만족하면 False를 반환한다.
    fields,
    coords,
    featuretype,
    positionrange,
    strand,
    attribute_filters,
    exclude_featuretype,
    exclude_positionrange,
    exclude_strand,
    exclude_attribute_filters,
):
    if featuretype and fields[TYPE].lower() != featuretype.lower():
        return False
    if positionrange:
        if coords is None:
            return False
        start, end = coords
        min_pos, max_pos = positionrange
        if not (start > min_pos and end < max_pos):
            return False
    if strand and fields[STRAND] != strand:
        return False
    if attribute_filters:
        attrs_lower = parse_attribute_column(fields[ATTRIBUTES])
        for key, op, value in attribute_filters:
            if not attribute_matches(attrs_lower, key, op, value):
                return False

    if exclude_featuretype and fields[TYPE].lower() == exclude_featuretype.lower():
        return False
    if exclude_positionrange and coords is not None:
        start, end = coords
        min_pos, max_pos = exclude_positionrange
        if start > min_pos and end < max_pos:
            return False
    if exclude_strand and fields[STRAND] == exclude_strand:
        return False
    if exclude_attribute_filters:
        attrs_lower = parse_attribute_column(fields[ATTRIBUTES])
        for key, op, value in exclude_attribute_filters:
            if attribute_matches(attrs_lower, key, op, value):
                return False
    return True


def main(): # 전체적인 워크플로우를 관리한다. argparse로 인자를 파싱하고, 입력 파일을 읽고, 각 줄을 필터링하고, 결과를 출력 파일에 쓴다. 마지막에 통계 정보를 출력한다.
    args = parse_args()

    if not os.path.isfile(args.gff_filepath):
        sys.exit(f"Error: file not found: {args.gff_filepath}")

    if args.outputdir and not os.path.isdir(args.outputdir):
        sys.exit(f"Error: output directory not found: {args.outputdir}")

    positionrange = parse_positionrange(args.positionrange) if args.positionrange else None
    attribute_filters = (
        [parse_attribute_filter(raw) for raw in args.attributes] if args.attributes else []
    )
    exclude_positionrange = (
        parse_positionrange(args.exclude_positionrange) if args.exclude_positionrange else None
    )
    exclude_attribute_filters = (
        [parse_attribute_filter(raw) for raw in args.exclude_attributes]
        if args.exclude_attributes
        else []
    )

    any_filter = bool(
        args.featuretype
        or positionrange
        or args.strand
        or attribute_filters
        or args.exclude_featuretype
        or exclude_positionrange
        or args.exclude_strand
        or exclude_attribute_filters
    )
    if not any_filter:
        print("No filter option added.")
        return

    needs_coords = bool(positionrange or exclude_positionrange)
    total_lines = 0
    skipped_lines = 0
    invalid_coord_lines = 0
    matched_lines = []

    with open(args.gff_filepath, newline="") as infile:
        for lineno, raw_line in enumerate(infile, start=1):
            line = raw_line.rstrip("\r\n")
            if not line:
                continue
            total_lines += 1
            fields = line.split("\t")
            if len(fields) != NUM_COLUMNS:
                print(f"Warning: line {lineno} does not have {NUM_COLUMNS} columns, skipping")
                skipped_lines += 1
                continue
            coords = parse_coords(fields) if needs_coords else None
            if needs_coords and coords is None:
                print(f"Warning: line {lineno} has non-numeric start/end, position filters ignored for this line")
                invalid_coord_lines += 1
            if line_matches(
                fields,
                coords,
                args.featuretype,
                positionrange,
                args.strand,
                attribute_filters,
                args.exclude_featuretype,
                exclude_positionrange,
                args.exclude_strand,
                exclude_attribute_filters,
            ):
                matched_lines.append(line)

    output_path = build_output_path(
        args.gff_filepath,
        args.featuretype,
        positionrange,
        args.strand,
        attribute_filters,
        args.exclude_featuretype,
        exclude_positionrange,
        args.exclude_strand,
        exclude_attribute_filters,
        args.outputdir,
    )

    if os.path.exists(output_path):
        answer = input(f"'{output_path}' already exists. Overwrite? [y/N] ").strip().lower()
        if answer != "y":
            print("Aborted, no file written.")
            return

    with open(output_path, "w") as outfile:
        for line in matched_lines:
            outfile.write(line + "\n")

    print(f"Total lines read:      {total_lines}")
    print(f"Lines skipped:         {skipped_lines}")
    if needs_coords:
        print(f"Lines w/ bad coords:   {invalid_coord_lines}")
    print(f"Lines matched:         {len(matched_lines)}")
    print(f"Output written to:     {output_path}")


if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] [-featuretype FEATURETYPE]
                             [-positionrange POSITIONRANGE] [-strand {+,-}]
                             [-outputdir OUTPUTDIR]
                             [-attribute KEY=VALUE|KEY~SUBSTRING|KEY]
                             [-exclude-featuretype EXCLUDE_FEATURETYPE]
                             [-exclude-positionrange EXCLUDE_POSITIONRANGE]
                             [-exclude-strand {+,-}]
                             [-exclude-attribute KEY=VALUE|KEY~SUBSTRING|KEY]
                             gff_filepath
ipykernel_launcher.py: error: the following arguments are required: gff_filepath


SystemExit: 2

/opt/conda/envs/sbml/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Loading GFF Files with pandas

Loading a GFF file with pandas requires a few non-obvious options. Use Claude Code to figure out the correct `read_csv` call for `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff`.

> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [ ]:
# pd.read_csv()는 파일을 tsb-separated text (sep="\t")로 읽는다. 클로드가 파일을 미리 읽어보니 헤더가 없었기 때문에 header=None을 지정하여, 0행 부터 데이터라고 pandas에게 전달하였다. pandas는 데이터에서 strand나 phase같은 임의적인 컬럼명을 유추하지 못하기 때문에 names=gff_cols는 GFF3의 세부사항에서 가져왔다.
# sep="\t" - GFF 파일은 탭으로 구분되어 있으므로, sep="\t"를 지정하여 pandas에게 알려준다.
# lineterminator="\n" - GFF 파일은 줄바꿈으로 구분되어 있으므로, lineterminator="\n"를 지정하여 pandas에게 알려준다. (follow-up 1에서 제거됨)
# follow-up 1: eliminated the lineterminator argument, as pandas can handle the line endings automatically.
import pandas as pd

gff_cols = ["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]

df = pd.read_csv(
    "/workspaces/intern-claude-training/data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff",
    sep="\t",
    header=None,
    names=gff_cols,
)

print(df.head())

       seqid       source  type  start   end score strand phase  \
0  NC_000913  Ecocyc_14.1  gene    190   255     .      +     .   
1  NC_000913  Ecocyc_14.1  gene    337  2799     .      +     .   
2  NC_000913  Ecocyc_14.1  gene   2801  3733     .      +     .   
3  NC_000913  Ecocyc_14.1  gene   3734  5020     .      +     .   
4  NC_000913  Ecocyc_14.1  gene   5234  5530     .      +     .   

                                          attributes  
0  locus_tag=b0001;gene=thrL;feature=gene;product...  
1  locus_tag=b0002;gene=thrA;feature=gene;product...  
2  locus_tag=b0003;gene=thrB;feature=gene;product...  
3  locus_tag=b0004;gene=thrC;feature=gene;product...  
4  locus_tag=b0005;gene=yaaX;feature=gene;product...  


### Exercise 2 — Debugging: Loading and Indexing

The script below has **two bugs**. Use Claude Code to diagnose and fix them.

Workflow:
1. Read the script and try to spot the bugs yourself first.
2. Paste the script into Claude Code and ask it to find the bugs.
3. Use `/debug` in the Claude Code terminal if you get stuck.
4. Write the corrected version in the empty code cell below the broken one.

> **Hint:** there is one bug in how the file is loaded, and one bug in how a row is accessed.

> **Explain it:** after fixing it, write 1–2 sentences on *what each bug was and why your fix works*. Understanding the bug matters more than the patch.

In [4]:
import pandas as pd

# Load GFF file
df = pd.read_csv('data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff', sep='\t')    # 경로 문제? 경로 앞에 "../" 추가하니 해결됨.

# Filter for forward strand genes
forward = df[df['strand'] == '+']   # KeyError = 'strand': 해당 GFF 파일은 header가 없고 바로 데이터로 시작한다. pandas는 첫째줄을 헤더로 인식하게 되며, 'strand' 컬럼이 없다고 판단한다.

# Get the 5th feature
fifth = forward.iat[5]  # 

print(f"5th forward feature starts at position {fifth['start']}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff'

In [ ]:
import pandas as pd

gff_cols = ["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]

# Load GFF file
df = pd.read_csv('../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff', sep='\t', header=None, names=gff_cols) # 경로 문제? 경로 앞에 "../" 추가하니 해결됨.

# Filter for forward strand genes
forward = df[df['strand'] == '+']   # KeyError = 'strand': 해당 GFF 파일은 header가 없고 바로 데이터로 시작한다. pandas는 첫째줄을 헤더로 인식하게 되며, 'strand' 컬럼이 없다고 판단한다.

# Get the 5th feature (row index 4, since forward.iloc is 0-indexed and .iat needs a (row, col) pair)
fifth = forward.iloc[4] # iat은 (row, col) pair를 필요로 하며, 하나의 scalar value를 반환한다. 따라서 5번째 forward feature 줄 전체를 가져오려면 iloc[4]를 사용해야 한다.

print(f"5th forward feature starts at position {fifth['start']}")

5th forward feature starts at position 5234


## `iterrows()` — When and Why

`iterrows()` iterates over a DataFrame row by row. Use Claude Code to understand when it is appropriate and when to avoid it.

### Exercise 3a — When to use it (concept)

After getting Claude Code's explanation, write a **2-sentence answer** in the markdown cell below:
- Sentence 1: when you **should** use `iterrows()`.
- Sentence 2: when you **should avoid** it and what to use instead.

### Exercise 3b — Use it on real data (hands-on)

Now actually iterate. Using the annotation DataFrame you loaded above, **find the gene whose transcription start site (TSS) is closest to position 1,000,000** on the genome.

- For a `+`-strand gene the TSS is its `start`; for a `-`-strand gene the TSS is its `end`.
- Loop over rows with `df.iterrows()`, compute each gene's distance to 1,000,000, and report the closest gene's name (the `gene=` value in the attribute column) and its distance.

Write the working script in the code cell below. When it runs, ask Claude Code how you would do the *same* computation **without** `iterrows()` (vectorized) — this is the practical version of the concept from Exercise 2a.


> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [3]:
# follow-up 1: added timing around the iterrows() loop, printed at the end
# follow-up 2: also report the closest feature's TSS position, not just its distance
import time

import pandas as pd

GFF_PATH = "/workspaces/intern-claude-training/data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff"
TARGET = 1_000_000

gff_cols = ["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]

# Count lines in the GFF file before loading with pandas
with open(GFF_PATH) as f:
    total_lines = sum(1 for _ in f)
print(f"Total lines in GFF file: {total_lines}")

df = pd.read_csv(GFF_PATH, sep="\t", header=None, names=gff_cols)
assert len(df) == total_lines, "line count / row count mismatch"

closest_gene_name = None
closest_distance = None
closest_tss = None
next_milestone = 10

start_time = time.perf_counter()
for idx, row in df.iterrows():
    tss = row["start"] if row["strand"] == "+" else row["end"]
    distance = abs(tss - TARGET)

    if closest_distance is None or distance < closest_distance:
        closest_distance = distance
        closest_tss = tss
        closest_gene_name = row["attributes"].split("gene=")[1].split(";")[0]

    rows_done = idx + 1
    pct_done = 100 * rows_done / total_lines
    if pct_done >= next_milestone:
        print(f"Progress: {next_milestone}% ({rows_done}/{total_lines} rows)")
        next_milestone += 10

elapsed = time.perf_counter() - start_time

print(f"Closest gene to position {TARGET}: {closest_gene_name}, TSS={closest_tss}, distance={closest_distance}")
print(f"Time spent on iterrows() loop: {elapsed:.4f} seconds")


Total lines in GFF file: 4595
Progress: 10% (460/4595 rows)
Progress: 20% (919/4595 rows)
Progress: 30% (1379/4595 rows)
Progress: 40% (1838/4595 rows)
Progress: 50% (2298/4595 rows)
Progress: 60% (2757/4595 rows)
Progress: 70% (3217/4595 rows)
Progress: 80% (3676/4595 rows)
Progress: 90% (4136/4595 rows)
Progress: 100% (4595/4595 rows)
Closest gene to position 1000000: ycbT, TSS=1001030, distance=1030
Time spent on iterrows() loop: 0.1724 seconds


> **Answer**
>
> iterrows()는 데이터가 크지 않아서 느린 속도가 문제되지 않을 때, 한줄씩 읽어들이는 행마다 외부 API 호출, 파일 I/O가 필요하거나, 이전 행의 계산 결과가 다음 행에 영향을 줄 때처럼, 여러 컬럼을 동시에 참조하면서 로직이 단순 조건 선택을 넘어설 때 사용한다.
> 
> 각 행의 결과가 오직 그 행의 컬럼 값들로만 결정되어 조건선택(`np.where`, `np.select`, `pd.cut`)/산술(`df['a']-df['b']`)/최솟값·최댓값 탐색(`idxmin()`, `idxmax()`)/필터링(`df[mask]`)/문자열 처리(`.str` 접근자) 등을 컬럼 전체에 동시에 적용할 수 있다면, iterrows()는 피해야 한다.

## End of Session

Before closing, run `/log` in the Claude Code terminal to save a summary of your session.

This creates a record of the prompts you used and the scripts you generated — useful for your own reference and for lab documentation.

## Git — Commit Your Work

Every session ends with a commit. Run the commands below in your terminal (not in this notebook).

Write your own commit message following the format: `feat(module2): <what you did>`.  
If you're unsure what to write, ask Claude Code to suggest one based on what you worked on.

In [1]:
# Run these in the terminal (not here):
# git add notebooks/
# git commit -m "..."   # write your own commit message; use Claude Code to help if needed
# git push
